In [1]:
# Imports
import os
import json
import textwrap
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display, JSON
from week1.scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

# Sales Intelligence Brief Generator

## What this does

A variation of the Day 5 brochure generator — instead of creating marketing material *for* a company, this tool generates an **internal sales brief *about* a target company**, helping a salesperson prepare for outreach.

## Pipeline

```
Target URL
    │
    ▼
Step 1 — Scrape website + filter relevant links (gpt-5-nano)
    │
    ▼
Step 2 — Extract structured company profile as JSON (gpt-5-nano)
    │
    ▼
Step 3 — Generate sales brief streamed in markdown (gpt-4.1-mini)
```

> **Note on model choice:** `gpt-5-nano` is used for structured extraction tasks (cheaper, faster). `gpt-4.1-mini` is used for the final generation step where quality matters more.

In [2]:
# Initializate and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key?")

MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


## Setup

Load API key from `.env` and initialize the OpenAI client.

In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to generate an internal sales brief about a target company — the kind
  of document a salesperson would use before an outreach call. E.g "About Us", "Careers Job pages", "Product services pages" etc...
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https:/|/another.full.url/careers"}
    ]
}
"""

## Step 1 — Scrape website & filter relevant links

Uses `gpt-5-nano` with **one-shot prompting** to intelligently filter which links from a webpage are worth scraping (about pages, product pages, careers, etc.), returning structured JSON.

This avoids hardcoding URL patterns — the LLM understands context and nuance that simple string matching can't.

In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links to generate an internal sales brief about a target company.
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [5]:
# select relevant links
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [6]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [7]:
profile_system_prompt = """
You are a business analyst. Given the content of a company's website, extract key information and respond in this exact JSON format:
{
      "industry": "...",
      "company_size": "startup / mid-size / enterprise",
      "key_products": ["product1", "product2"],
      "target_customers": ["segment1", "segment2"],
      "tech_mentions": ["tool1", "tool2"],
      "pain_points_they_solve": ["pain1", "pain2"]
  }
"""

## Step 2 — Extract structured company profile

Uses `gpt-5-nano` with `response_format={"type": "json_object"}` (JSON mode) to extract a structured company profile from raw scraped text.

Content is truncated to 5,000 characters to stay within token limits and keep costs low — the landing page and first few relevant pages are usually enough to infer the key fields.

In [8]:
def extract_company_profile(url_content):
    print("Extracting company profile...")
    response = openai.chat.completions.create(
        model=MODEL,
        messages = [
            {"role": "system", "content": profile_system_prompt},
            {"role": "user", "content": url_content[:5_000]}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    json_response = json.loads(result)
    print(f"Profile extracted: {json_response['industry']} | {json_response['company_size']}")
    return json_response

In [9]:
sales_brief_system_prompt = """
You are an expert sales strategist. Given a target company's profile and a product being sold,
generate a concise internal sales brief for a salesperson preparing for outreach.
Structure your response in markdown with these sections:
## Who They Are                                                                                                                        
## Why They Need This Product
## Suggested Talking Points
## Sample Cold Email Opening Line
"""

## Step 3 — Generate the sales brief

Uses `gpt-4.1-mini` with **streaming** to generate a focused 1-page sales brief in markdown.

The brief is personalized by combining two inputs:
- `company_profile` — structured JSON about the target company (from Step 2)
- `your_product` — a plain text description of the product being sold

Passing structured JSON rather than raw scraped text gives the LLM cleaner, more focused input — producing more relevant output.

In [10]:
def stream_sales_brief(your_product, company_profile):
      user_prompt = f"""
  Here is the target company's profile:
  {json.dumps(company_profile, indent=2)}

  Here is the product we are selling:
  {your_product}

  Generate a focused 1-page sales brief.
  """
      stream = openai.chat.completions.create(
          model="gpt-4.1-mini",
          messages=[
              {"role": "system", "content": sales_brief_system_prompt},
              {"role": "user", "content": user_prompt}
          ],
          stream=True
      )
      response = ""
      display_handle = display(Markdown(""), display_id=True)
      for chunk in stream:
          response += chunk.choices[0].delta.content or ""
          update_display(Markdown(response), display_id=display_handle.display_id)

In [11]:
your_product = textwrap.dedent("""
    AI-powered code review tool that detects bugs, security vulnerabilities,
    and performance issues in pull requests. Integrates with GitHub and GitLab.
    Targeted at engineering teams of 10+ developers.
    """).strip()

## Run it

Set `your_product` to describe what you're selling, then call `run_pipeline` with the target company's URL.

In [12]:
def run_pipeline(target_url, your_product_description):
    content = fetch_page_and_all_relevant_links(target_url)
    company_profile = extract_company_profile(content)
    stream_sales_brief(your_product_description, company_profile)
    JSON(company_profile)

In [13]:
run_pipeline("https://www.worldoffice.com.co/", your_product)

Extracting company profile...
Profile extracted: Cloud accounting and invoicing software / ERP | mid-size


## Who They Are  
A mid-sized SaaS company specializing in cloud accounting and ERP solutions tailored for Colombian SMEs. Their flagship offerings include DIAN-compliant electronic invoicing, automated accounting, payroll, POS, inventory management, and mobile app access, all delivered via a Microsoft-integrated cloud platform with robust APIs. They serve businesses needing compliance and automation to simplify financial operations while enabling remote accessibility and real-time updates.

## Why They Need This Product  
- Their development team likely manages complex integrations (e.g., DIAN compliance, cloud APIs, mobile apps) and must maintain high code quality to ensure secure, reliable financial software.  
- Manual code review or quality gaps could lead to bugs, performance issues, or security vulnerabilities that directly impact client compliance and operational trust.  
- AI-powered code review can streamline their engineering workflow by automatically detecting issues early in pull requests, reducing manual effort and accelerating release cycles.  
- Integration with GitHub/GitLab fits naturally into their existing developer tools, supporting their team's collaboration at scale (10+ engineers).  
- Enhancing code security and performance protects their competitive edge in a regulated, high-stakes industry (finance and tax compliance).

## Suggested Talking Points  
- How automated AI code review can improve detection of critical bugs and security vulnerabilities in real time, preventing costly downstream issues in financial software.  
- Seamless integration with GitHub and GitLab to embed quality checks directly into existing developer workflows.  
- Time and cost savings through reduced manual code review effort and faster, safer deployments.  
- How improved code reliability aids compliance with DIAN and enhances customer trust.  
- Case examples of how other mid-sized SaaS/ERP/fintech firms accelerated development velocity and reduced risk with AI code review tools.

## Sample Cold Email Opening Line  
Hi [Name], given your focus on delivering compliant, reliable cloud accounting solutions to Colombian SMEs, I wanted to share how our AI-powered code review tool helps engineering teams identify bugs and security risks early, so you can accelerate releases without compromising quality.